In [1]:
# [1/5] HEALNet 설치
import os, sys

HEALNET_DIR = "/kaggle/working/healnet"

if not os.path.exists(HEALNET_DIR):
    os.system(f"git clone https://github.com/konst-int-i/healnet.git {HEALNET_DIR}")

os.chdir(HEALNET_DIR)
os.system(f"{sys.executable} -m pip install -e . -q")

# loaders.py의 openslide import를 try/except로 감싸기 (로컬 환경 호환)
loaders_path = os.path.join(HEALNET_DIR, "healnet/etl/loaders.py")
with open(loaders_path) as f:
    src = f.read()

old = "from openslide import OpenSlide"
new = "try:\n    from openslide import OpenSlide\nexcept ImportError:\n    OpenSlide = None"
if old in src:
    with open(loaders_path, "w") as f:
        f.write(src.replace(old, new))
    print("loaders.py 패치 완료")

print("설치 완료")

loaders.py 패치 완료
설치 완료


In [2]:
# [2/5] 합성 데이터 생성
# 실제 TCGA-KIRP 데이터와 동일한 shape을 갖는 랜덤 텐서를 생성합니다.
# 이미지(WSI) 데이터 다운로드 없이 멀티모달 HEALNet의 동작을 확인합니다.
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

# ── 데이터 크기 설정 ──────────────────────────────────────────────────────
N          = 200    # 샘플 수 (실제 KIRP: 284)
N_OMIC     = 1587   # 오믹 피처 수 (실제 KIRP: 1587)
N_PATCHES  = 256    # WSI 당 패치 수 (실제: 수백~수천)
PATCH_DIM  = 1024   # 패치 피처 차원 (실제: ResNet50 → 1024-dim)
N_BINS     = 4      # 생존 구간 수 (논문 설정과 동일)

# ── 모달리티 1: 오믹 데이터 ────────────────────────────────────────────────
# 실제: tcga_kirp_all_clean.csv에서 로드한 유전체 발현 데이터
# shape: (N, channels=1, n_omic_features)
omic_tensor = torch.rand(N, 1, N_OMIC)

# ── 모달리티 2: WSI 패치 피처 ──────────────────────────────────────────────
# 실제: .svs 이미지에서 CLAM으로 패치 추출 → ResNet50으로 피처 변환한 .pt 파일
# shape: (N, n_patches, patch_feature_dim)
wsi_tensor = torch.rand(N, N_PATCHES, PATCH_DIM)

# ── 생존 분석 라벨 ───────────────────────────────────────────────────────
# censorship: 0=사망(이벤트 발생), 1=중도 절단(생존 추적 종료)
censorship   = torch.bernoulli(torch.full((N,), 0.4))          # 40%가 이벤트 발생
event_time   = torch.FloatTensor(N).uniform_(1, 200)           # 생존 기간 (개월)
y_disc       = torch.randint(0, N_BINS, (N,))                  # 이산화된 생존 구간

print(f"오믹 텐서       : {omic_tensor.shape}  (N, channels, omic_features)")
print(f"WSI 패치 피처   : {wsi_tensor.shape} (N, n_patches, patch_dim)")
print(f"censorship      : {censorship.shape}")
print(f"event_time      : {event_time.shape}")
print(f"y_disc (라벨)   : {y_disc.shape}")

오믹 텐서       : torch.Size([200, 1, 1587])  (N, channels, omic_features)
WSI 패치 피처   : torch.Size([200, 256, 1024]) (N, n_patches, patch_dim)
censorship      : torch.Size([200])
event_time      : torch.Size([200])
y_disc (라벨)   : torch.Size([200])


In [3]:
# [3/5] Dataset & DataLoader 구성
from torch.utils.data import Dataset, DataLoader, random_split
from typing import List, Optional, Tuple

# MMDataset: healnet.etl에서 가져오는 대신 직접 정의
# (loaders.py가 openslide를 import하므로 로컬 환경에서 오류 발생)
class MMDataset(Dataset):
    """Generic torch dataset for supervised multi-modal data."""
    def __init__(self, tensors: List[torch.Tensor], target: Optional[torch.Tensor] = None):
        self.tensors = tensors
        self.target  = target

    def __getitem__(self, idx) -> Tuple:
        if self.target is None:
            return [t[idx] for t in self.tensors]
        return [t[idx] for t in self.tensors], self.target[idx]

    def __len__(self):
        return self.tensors[0].size()[0]

# 타겟은 (N, 3) 형태로 패킹 — [censorship, event_time, y_disc]
target = torch.stack([censorship, event_time, y_disc.float()], dim=1)

dataset = MMDataset([omic_tensor, wsi_tensor], target)

# 70 / 15 / 15 분할
n_train = int(N * 0.70)
n_val   = int(N * 0.15)
n_test  = N - n_train - n_val
train_set, val_set, test_set = random_split(dataset, [n_train, n_val, n_test])

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=16, shuffle=False)

print(f"Train: {len(train_set)}  Val: {len(val_set)}  Test: {len(test_set)}")

# 배치 shape 확인
[omic_batch, wsi_batch], target_batch = next(iter(train_loader))
print(f"\n배치 shape 확인")
print(f"  omic  : {omic_batch.shape}")
print(f"  wsi   : {wsi_batch.shape}")
print(f"  target: {target_batch.shape}  → [censorship, event_time, y_disc]")

Train: 140  Val: 30  Test: 30

배치 shape 확인
  omic  : torch.Size([16, 1, 1587])
  wsi   : torch.Size([16, 256, 1024])
  target: torch.Size([16, 3])  → [censorship, event_time, y_disc]


In [4]:
# [4/5] HEALNet 모델 초기화 및 학습
from healnet.models import HealNet
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스: {device}")

# ── 각 모달리티의 spatial_axes와 channel_dims 계산 ──────────────────────────
# HEALNet 입력 형식: (batch, *spatial_dims, channels)
# - 오믹  (batch, 1, 1587)  → spatial_axes=1, channels=1587
# - WSI   (batch, 256, 1024) → spatial_axes=1, channels=1024
spatial_axes = []
channel_dims = []
for tensor in [omic_batch, wsi_batch]:
    b, *spatial, c = tensor.shape
    spatial_axes.append(len(spatial))
    channel_dims.append(c)

print(f"spatial_axes : {spatial_axes}")
print(f"channel_dims : {channel_dims}")

# ── 모델 생성 (논문 KIRP 하이퍼파라미터 기반) ─────────────────────────────
model = HealNet(
    n_modalities        = 2,             # 오믹 + WSI
    channel_dims        = channel_dims,
    num_spatial_axes    = spatial_axes,
    out_dims            = N_BINS,        # 생존 구간 수
    l_c                 = 25,            # latent channels (논문 HEALNet 값)
    l_d                 = 119,           # latent dims    (논문 HEALNet 값)
    depth               = 5,            # KIRP depth
    x_heads         = 1,
    l_heads        = 8,
    cross_dim_head      = 27,
    latent_dim_head     = 113,
    attn_dropout        = 0.32,
    ff_dropout          = 0.05,
    fourier_encode_data = True,
    snn                 = True,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"\n모델 파라미터 수: {total_params:,}")

# ── 학습 설정 ─────────────────────────────────────────────────────────────
# 실제 HEALNet은 NLL survival loss를 사용하나,
# 튜토리얼에서는 이산화된 구간(y_disc)에 CrossEntropyLoss로 대체합니다.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10

print(f"\n{'Epoch':>6}  {'Train Loss':>11}  {'Val Loss':>10}")
print("-" * 34)

for epoch in range(1, EPOCHS + 1):
    # ── 학습 ──
    model.train()
    train_loss = 0.0
    for (omic_b, wsi_b), tgt_b in train_loader:
        omic_b = omic_b.to(device)
        wsi_b  = wsi_b.to(device)
        y      = tgt_b[:, 2].long().to(device)   # y_disc

        logits = model([omic_b, wsi_b])
        loss   = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # ── 검증 ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for (omic_b, wsi_b), tgt_b in val_loader:
            omic_b = omic_b.to(device)
            wsi_b  = wsi_b.to(device)
            y      = tgt_b[:, 2].long().to(device)
            logits = model([omic_b, wsi_b])
            val_loss += criterion(logits, y).item()

    print(f"{epoch:>6}  {train_loss/len(train_loader):>11.4f}  {val_loss/len(val_loader):>10.4f}")

print("\n학습 완료")

사용 디바이스: cpu
spatial_axes : [1, 1]
channel_dims : [1587, 1024]

모델 파라미터 수: 5,527,323

 Epoch   Train Loss    Val Loss
----------------------------------
     1       1.9217      1.6466
     2       1.3641      1.3526
     3       1.2977      1.4579
     4       1.2019      1.3930
     5       0.6998      2.4735
     6       0.3343      2.2825
     7       0.1946      2.3566
     8       0.0779      3.5182
     9       0.1932      3.3682
    10       0.0791      3.3983

학습 완료


In [5]:
# [5/5] 누락 모달리티 시연
# HEALNet은 추론 시 모달리티가 없어도 None을 전달하면 해당 모달리티를 건너뜁니다.
# 이것이 Table 2에서 보여주는 HEALNet의 핵심 강점입니다.

model.eval()
[omic_b, wsi_b], tgt_b = next(iter(test_loader))
omic_b = omic_b.to(device)
wsi_b  = wsi_b.to(device)

with torch.no_grad():
    # 케이스 1: 두 모달리티 모두 존재
    logits_full  = model([omic_b, wsi_b])

    # 케이스 2: WSI 없음 (오믹만)
    # → 실제 TCGA에서 이미지가 없는 환자 샘플에 해당
    logits_omic  = model([omic_b, None], verbose=False)

    # 케이스 3: 오믹 없음 (WSI만)
    # → 유전체 데이터가 없는 경우
    logits_wsi   = model([None, wsi_b],  verbose=False)

print("누락 모달리티 시나리오별 예측 분포 (첫 번째 샘플, softmax 확률)")
print(f"  오믹 + WSI (전체)  : {torch.softmax(logits_full[0], dim=0).cpu().numpy().round(3)}")
print(f"  오믹만             : {torch.softmax(logits_omic[0], dim=0).cpu().numpy().round(3)}")
print(f"  WSI만              : {torch.softmax(logits_wsi[0],  dim=0).cpu().numpy().round(3)}")

print()
print("※ 합성 데이터이므로 수치 자체는 의미 없음")
print("  오믹 + WSI ≠ 오믹만 → 두 모달리티가 실제로 융합되어 예측에 영향을 줌")
print("  오믹만/WSI만으로도 예측 가능 → 누락 모달리티 강건성 확인")

누락 모달리티 시나리오별 예측 분포 (첫 번째 샘플, softmax 확률)
  오믹 + WSI (전체)  : [0.007 0.029 0.946 0.019]
  오믹만             : [0.035 0.074 0.399 0.492]
  WSI만              : [0.004 0.008 0.986 0.002]

※ 합성 데이터이므로 수치 자체는 의미 없음
  오믹 + WSI ≠ 오믹만 → 두 모달리티가 실제로 융합되어 예측에 영향을 줌
  오믹만/WSI만으로도 예측 가능 → 누락 모달리티 강건성 확인
